# WaterSIC: Information-Theoretically (Near) Optimal Linear Layer Quantization

> **Paper:** Lifar, Savkin, Ordentlich, Polyanskiy — *WaterSIC: information-theoretically (near) optimal linear layer quantization*, arXiv:2603.04956v1 (Mar 2026)

---

## 0. Motivation & Overview

Post-training quantization (PTQ) converts a dense linear layer $y = Wx$ to low precision without retraining. The standard metric is **weighted mean-squared error (WMSE)**:

$$\min_{\hat{W}} \; \mathbb{E}\|\hat{W}x - Wx\|^2 = \min_{\hat{W}} \; \text{tr}\bigl((\hat{W}-W)^\top (\hat{W}-W)\,\Sigma_X\bigr)$$

where $\Sigma_X = \mathbb{E}[xx^\top]$ is the input activation covariance (estimated from a calibration set).

The paper makes two core contributions:

1. **Theoretical gap:** GPTQ uses a uniform quantization grid spacing $\varepsilon$ for every column of $W$. The paper proves this can have an **arbitrarily large gap** to the information-theoretic (IT) optimum.

2. **WaterSIC algorithm:** By assigning *different* grid spacings $\varepsilon_i$ to different input-feature columns (via the **Water-filling** principle), WaterSIC closes this gap to at most **0.255 bits** from the IT limit, *uniformly* over all possible $\Sigma_X$.

---

## 1. Mathematical Background

### 1.1 Information-Theoretic Distortion Limits

Let $\Sigma_X$ have eigenvalues $\lambda_1, \dots, \lambda_n$. For a weight matrix $W$ with i.i.d. Gaussian entries and target coding rate $R$ bits-per-entry, the **IT-optimal distortion** (high-rate regime) is:

$$D^*(R) \approx |\Sigma_X|^{1/n} \cdot 2^{-2R}$$

where $|\Sigma_X|^{1/n} = \left(\prod_i \lambda_i\right)^{1/n}$ is the **geometric mean** of eigenvalues.

In contrast, GPTQ with uniform grid spacing $\varepsilon$ achieves:

$$D_{\text{GPTQ}}(R) \approx \frac{1}{n}\text{tr}(\Sigma_X) \cdot 2^{-2R} = \bar{\lambda}_A \cdot 2^{-2R}$$

where $\bar{\lambda}_A = \frac{1}{n}\sum_i \lambda_i$ is the **arithmetic mean**. By the AM–GM inequality, $\bar{\lambda}_A \geq \bar{\lambda}_G$, so GPTQ is always suboptimal — and the gap can be unbounded when eigenvalues are skewed.

### 1.2 Water-filling Rate Allocation (Section 3 of paper)

The classical (reverse) water-filling solution allocates grid spacing per-coordinate as:

$$\boxed{\varepsilon_i = \min\!\left(\sqrt{\frac{12\,\tau}{\lambda_i}},\; 1\right)}$$

where $\tau$ is a **water level** chosen so the total rate budget $\sum_i R_i = n \cdot R_{\text{avg}}$ is met. The equivalent rate per coordinate is:

$$R_i = \frac{1}{2}\log_2\!\left(\frac{\lambda_i}{\tau}\right)^+ \qquad (\text{clamped to } [b_{\min}, b_{\max}])$$

Intuitively: **high-importance directions** (large $\lambda_i$) get **finer grids** (small $\varepsilon_i$, more bits); low-importance directions get coarser grids or are quantized to zero.

The gain over uniform allocation is given by the **AM–GM gap**:

$$\frac{D_{\text{GPTQ}}}{D_{\text{WaterSIC}}} = \frac{\bar{\lambda}_A}{\bar{\lambda}_G} \geq 1$$

### 1.3 Successive Interference Cancellation (SIC)

WaterSIC inherits GPTQ's **SIC update loop** but applies per-column grid spacings. For each column $j$ (processed in order), the algorithm:

1. Quantises weight column $j$ using grid spacing $\varepsilon_j$:
   $$\hat{w}_j = \varepsilon_j \cdot \text{round}(w_j / \varepsilon_j)$$

2. Computes the quantisation error for column $j$:
   $$\delta_j = w_j - \hat{w}_j$$

3. **Propagates the error** to all remaining columns $k > j$ via the inverse Hessian:
   $$w_k \leftarrow w_k - \delta_j \cdot \frac{[H^{-1}]_{j,k}}{[H^{-1}]_{j,j}}$$

This is the SIC step — earlier quantisation errors are cancelled out from downstream columns, analogous to interference cancellation in MIMO communications.

### 1.4 Algorithm Summary

```
INPUT:  Weight W ∈ R^{m×n}, calibration activations X ∈ R^{N×n},
        target average bits R_avg

1.  Compute Hessian: H = X^T X  (+ damping)
2.  Eigendecompose Σ_X → eigenvalues {λ_i}
3.  Water-fill: solve for τ s.t. mean(R_i) = R_avg
               ε_i = min(sqrt(12τ/λ_i), 1)
4.  Cholesky: H = L L^T  →  compute H^{-1} via cholesky_inverse
5.  For j = 1 … n:   (SIC loop, column by column)
       ŵ_j = ε_j · round(w_j / ε_j)      # quantise with water-filled ε
       δ_j = w_j - ŵ_j                    # quantisation error
       w_{j+1:} -= δ_j · H^{-1}[j, j+1:] / H^{-1}[j,j]   # SIC update

OUTPUT: Quantised weight W_hat
```

---

## 2. Setup & Imports

In [ ]:
# Standard library + PyTorch
import math
import warnings
from typing import Optional, Tuple, List, Dict

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device selection — falls back to CPU cleanly
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Core Primitives

### 3.1 Hessian Computation

The Hessian of the reconstruction loss $\|Wx - \hat{W}x\|^2_F$ with respect to $\hat{W}$ is:

$$H = X^\top X \quad (\text{or } H = \frac{2}{N} X^\top X \text{ if normalised})$$

We add a **Tikhonov damping** term $\lambda I$ (with $\lambda = \delta \cdot \text{mean}(\text{diag}(H))$) to ensure numerical stability of the Cholesky decomposition — this is standard in GPTQ and carried over in WaterSIC.

In [ ]:
def compute_hessian(
    X: torch.Tensor,
    damp_percent: float = 0.01,
) -> torch.Tensor:
    """
    Compute the Hessian H = X^T X from calibration activations,
    with Tikhonov damping for numerical stability.

    The reconstruction loss for a linear layer is:
        L = || (W - W_hat) X ||^2_F
    whose Hessian w.r.t. W_hat rows is H = X X^T (per-row),
    or equivalently H = X^T X when operating column-wise (GPTQ convention).

    Paper reference: Section 2 & 3 — Sigma_X = (1/N) X^T X is the
    sample covariance of the calibration activations.

    Args:
        X            : activations, shape (N, in_features).
                       Can also be (B, T, in_features) — will be reshaped.
        damp_percent : fraction of mean diagonal used for damping lambda.

    Returns:
        H : (in_features, in_features) Hessian, fp32, on same device as X.
    """
    if X.dim() == 3:                          # (B, T, C) -> (B*T, C)
        X = X.reshape(-1, X.shape[-1])

    X = X.to(torch.float32)                   # accumulate in fp32
    N, C = X.shape

    # H = X^T X  (C x C)
    H = X.t().matmul(X)                       # efficient: single BLAS SGEMM

    # Damping: H += lambda * I,  lambda = damp_percent * mean(diag(H))
    # Prevents Cholesky failure when input channels are collinear.
    damp = damp_percent * H.diagonal().mean()
    H.diagonal().add_(damp)

    return H

### 3.2 Water-filling Bit / Grid-Spacing Allocation

Given eigenvalues $\{\lambda_i\}$ of $\Sigma_X$ and a budget $R_{\text{avg}}$ bits/entry, we find water level $\tau$ via **bisection** such that:

$$\frac{1}{n}\sum_{i=1}^{n} \left[\frac{1}{2}\log_2\!\left(\frac{\lambda_i}{\tau}\right)\right]^{b_{\min}}_{b_{\max}} = R_{\text{avg}}$$

The corresponding grid spacing (paper Eq. for $\varepsilon_i$) is:

$$\varepsilon_i = \sqrt{\frac{\tau}{\lambda_i}} \quad \text{(before clamping)}$$

In practice we round $R_i$ to the nearest integer for hardware compatibility, though entropy coding in the paper allows fractional rates.

In [ ]:
def waterfilling_allocate(
    eigenvalues: torch.Tensor,
    target_avg_bits: float,
    b_min: float = 1.0,
    b_max: float = 8.0,
    bisect_iters: int = 128,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Water-filling rate allocation over channel importances.

    Solves for water-level tau such that:
        mean( clamp( 0.5 * log2(lambda_i / tau), b_min, b_max ) ) = target_avg_bits

    Then returns:
      - integer bits per channel (rounded, for INT quantiser)
      - continuous grid spacing eps_i = sqrt(tau / lambda_i)
        (paper Section 3, water-filling formula eps_i ∝ 1/sqrt(lambda_i))

    Args:
        eigenvalues     : 1-D tensor of non-negative channel importances, shape (C,).
                          Typically the diagonal of H = X^T X (or true eigenvalues).
        target_avg_bits : desired mean bit-width, e.g. 4.0.
        b_min           : hard floor on bits (default 1).
        b_max           : hard ceiling on bits (default 8).
        bisect_iters    : iterations for bisection search (128 ~ 1e-19 precision).

    Returns:
        bits_int : LongTensor of shape (C,), integer bits in [b_min, b_max].
        eps      : FloatTensor of shape (C,), continuous grid spacings.
    """
    device = eigenvalues.device
    lam = eigenvalues.to(torch.float64).clamp(min=1e-12)  # fp64 for precision
    log2_lam = lam.log2()                                  # log2(lambda_i)

    def _mean_bits(log2_tau: torch.Tensor) -> torch.Tensor:
        """Mean bits for a given log2(tau)."""
        # R_i = 0.5 * log2(lambda_i / tau) = 0.5 * (log2_lam - log2_tau)
        R = 0.5 * (log2_lam - log2_tau)
        return R.clamp(b_min, b_max).mean()

    # ----------------------------------------------------------------
    # Bisection: find log2_tau s.t. mean_bits == target_avg_bits.
    # All tensor ops → runs fully on-device, no Python loop over channels.
    # ----------------------------------------------------------------
    lo = torch.tensor(-200.0, device=device, dtype=torch.float64)  # low tau → many bits
    hi = torch.tensor( 200.0, device=device, dtype=torch.float64)  # high tau → few bits

    for _ in range(bisect_iters):
        mid = (lo + hi) * 0.5
        mb  = _mean_bits(mid)
        # mean_bits decreases as log2_tau increases
        lo = torch.where(mb > target_avg_bits, mid, lo)
        hi = torch.where(mb <= target_avg_bits, mid, hi)

    log2_tau_star = (lo + hi) * 0.5
    tau_star      = (log2_tau_star * math.log(2)).exp()      # back to linear

    # Continuous grid spacing: eps_i = sqrt(tau / lambda_i)
    # Paper: eps_i = min(sqrt(12*tau / lambda_i), 1), but we absorb the
    # '12' normalisation into the quantiser definition.
    eps = (tau_star / lam).sqrt().clamp(max=1.0).to(torch.float32)

    # Integer bits for hardware-friendly INT quantiser
    R_cont  = 0.5 * (log2_lam - log2_tau_star)
    bits_int = R_cont.clamp(b_min, b_max).round().long()

    return bits_int, eps


# Quick sanity check
lam_test = torch.tensor([0.1, 1.0, 5.0, 20.0, 100.0], device=DEVICE)
bits_test, eps_test = waterfilling_allocate(lam_test, target_avg_bits=4.0)
print("Sanity check — Water-filling on 5 channels, target 4 bits:")
for i, (l, b, e) in enumerate(zip(lam_test.tolist(), bits_test.tolist(), eps_test.tolist())):
    print(f"  ch {i}: lambda={l:6.1f}  ->  bits={b}  eps={e:.4f}")
print(f"  Actual mean bits = {bits_test.float().mean():.2f}  (target = 4.0)")

### 3.3 Scalar Quantiser

The paper uses an $\varepsilon$-grid (uniform scalar) quantiser:
$$Q_\varepsilon(v) = \varepsilon \cdot \text{round}(v/\varepsilon)$$

For integer-bit versions we use the standard asymmetric integer quantiser with scale derived from the bit-width.

In [ ]:
def quantize_eps(
    x: torch.Tensor,
    eps: float,
) -> torch.Tensor:
    """
    Epsilon-grid quantiser: Q_eps(x) = eps * round(x / eps).
    Paper Section 2: base quantiser on an epsilon-grid.
    """
    if eps <= 0:
        return x
    return (x / eps).round() * eps


def quantize_int(
    x: torch.Tensor,
    bits: int,
) -> torch.Tensor:
    """
    Asymmetric uniform integer quantiser to `bits` bits.
    Maps x to the nearest value on a 2^bits-level grid covering [x.min, x.max].
    Returns dequantised fp tensor (same shape/dtype as x).

    Used for hardware-friendly variant where bits must be an integer.
    """
    if bits >= 16:
        return x
    qmin, qmax = 0, 2 ** bits - 1
    x_min, x_max = x.min(), x.max()
    scale = (x_max - x_min).clamp(min=1e-8) / qmax
    zero  = (-x_min / scale).round().clamp(qmin, qmax)
    x_q   = ((x / scale) + zero).round().clamp(qmin, qmax)
    return ((x_q - zero) * scale).to(x.dtype)

---

## 4. WaterSIC Quantizer Class

The main class follows the three-phase structure described in the paper:

| Phase | Method | Paper Reference |
|-------|---------|----------------|
| 1. Calibration | `add_batch()` | Section 2: estimate $\Sigma_X$ |
| 2. Allocation | `_waterfill()` | Section 3: $\varepsilon_i \propto 1/\sqrt{\lambda_i}$ |
| 3. Quantisation | `quantize()` | Section 5/Alg. 1: SIC loop with per-column $\varepsilon_i$ |

In [ ]:
class WaterSICQuantizer:
    """
    WaterSIC: Water-filling Successive Interference Cancellation Quantizer.

    Improves upon GPTQ by replacing the fixed (uniform) quantization grid
    spacing with a per-column spacing derived from the water-filling solution
    over the Hessian eigenvalues.

    Key result (paper Thm 1 / Section 3):
        WaterSIC is within 0.255 bits/entry of the IT optimal distortion,
        uniformly over all Sigma_X, while GPTQ can be arbitrarily worse.

    Usage:
        q = WaterSICQuantizer(layer)
        q.add_batch(X)           # accumulate calibration activations
        W_q, info = q.quantize(target_avg_bits=4.0)
        # layer.weight is updated in-place; info contains diagnostics.
    """

    def __init__(
        self,
        layer: nn.Linear,
        damp_percent: float = 0.01,
        block_size:   int   = 128,
        actorder:     bool  = True,
        b_min:        int   = 1,
        b_max:        int   = 8,
        use_eps_grid: bool  = True,
    ):
        """
        Args:
            layer        : nn.Linear layer to quantise.
            damp_percent : Hessian diagonal damping fraction (default 0.01).
            block_size   : columns per Cholesky block (paper uses 128).
            actorder     : reorder columns by decreasing H diagonal (activation
                           order) — improves accuracy, standard in GPTQ.
            b_min, b_max : bit-width clamp range for water-filling.
            use_eps_grid : if True, use continuous eps-grid quantiser;
                           if False, use integer-bit quantiser.
        """
        self.layer        = layer
        self.damp_pct     = damp_percent
        self.block_size   = block_size
        self.actorder     = actorder
        self.b_min        = b_min
        self.b_max        = b_max
        self.use_eps_grid = use_eps_grid

        W = layer.weight.data                  # (out_features, in_features)
        self.rows, self.cols = W.shape
        dev = W.device

        # Running Hessian accumulator: H = sum_batches( X^T X )
        self.H      = torch.zeros(self.cols, self.cols, device=dev, dtype=torch.float32)
        self.n_samp = 0

    # ------------------------------------------------------------------
    # Phase 1: Calibration
    # ------------------------------------------------------------------

    def add_batch(self, X: torch.Tensor) -> None:
        """
        Accumulate Hessian statistics from a mini-batch of activations.

        Paper Section 2: H = X^T X estimated from calibration set.
        Accepts shape (N, C) or (B, T, C) — the latter is flattened.
        """
        if X.dim() == 3:
            X = X.reshape(-1, X.shape[-1])
        X = X.to(torch.float32)
        # Accumulate H = sum X_i^T X_i — will normalise in quantize()
        self.H      += X.t().matmul(X)
        self.n_samp += X.shape[0]

    # ------------------------------------------------------------------
    # Phase 2: Water-filling allocation (internal)
    # ------------------------------------------------------------------

    def _waterfill(
        self,
        h_diag: torch.Tensor,
        target_avg_bits: float,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Run water-filling on H diagonal (proxy for lambda_i) and return
        (integer bits per column, continuous eps per column).

        Paper Section 3: lambda_i are used to compute optimal per-column
        grid spacing eps_i = sqrt(tau / lambda_i).
        """
        return waterfilling_allocate(
            h_diag,
            target_avg_bits=target_avg_bits,
            b_min=self.b_min,
            b_max=self.b_max,
        )

    # ------------------------------------------------------------------
    # Phase 3: SIC quantisation loop
    # ------------------------------------------------------------------

    @torch.no_grad()
    def quantize(
        self,
        target_avg_bits: float = 4.0,
        per_column_alloc: bool = True,
    ) -> Tuple[torch.Tensor, Dict]:
        """
        Quantise layer.weight in-place using the WaterSIC algorithm.

        Paper Algorithm (Section 5):
          - Uniform ε: GPTQ baseline
          - Per-column ε from water-filling: WaterSIC

        Args:
            target_avg_bits  : mean bit-width budget.
            per_column_alloc : True = WaterSIC (water-filling per column)
                               False = GPTQ baseline (uniform bits)

        Returns:
            W_q  : quantised weight tensor (fp32).
            info : dict with diagnostics (bits per col, eps per col, etc.).
        """
        assert self.n_samp > 0, "Call add_batch() before quantize()."

        dev   = self.layer.weight.device
        orig_dtype = self.layer.weight.dtype

        # ---- Working copies in fp32 ----
        W = self.layer.weight.data.clone().float()  # (rows, cols)
        H = (self.H / self.n_samp).clone()          # normalised Hessian

        # ---- Damping (paper Section 5: standard GPTQ trick) ----
        damp = self.damp_pct * H.diagonal().mean()
        H.diagonal().add_(damp)

        # ---- Activation-order reordering ----
        # Reorder columns by decreasing H diagonal so the most important
        # features are quantised first and compensated most accurately.
        h_diag = H.diagonal().clone()
        if self.actorder:
            perm     = torch.argsort(h_diag, descending=True)
            inv_perm = torch.argsort(perm)
            W        = W[:, perm]
            H        = H[perm][:, perm]
            h_diag   = H.diagonal().clone()

        # ---- Water-filling (or uniform) bit/eps allocation ----
        if per_column_alloc:
            # WaterSIC: per-column eps from water-filling over H diagonal
            # Paper Section 3: eps_i = sqrt(tau / lambda_i)
            col_bits, col_eps = self._waterfill(h_diag, target_avg_bits)
        else:
            # GPTQ baseline: uniform bits, uniform eps (derived from bit-width)
            ub = int(round(target_avg_bits))
            col_bits = torch.full((self.cols,), ub, dtype=torch.long, device=dev)
            # Uniform eps: for INT-b, the step size relative to range is ~2^(-b)
            col_eps  = torch.full((self.cols,), 2 ** (-ub), dtype=torch.float32, device=dev)

        # ---- Cholesky decomposition of H for stable H^{-1} ----
        try:
            L     = torch.linalg.cholesky(H)       # H = L L^T
            H_inv = torch.cholesky_inverse(L)       # H^{-1} via back-substitution
        except torch.linalg.LinAlgError:
            warnings.warn("Cholesky failed; adding extra damping 1e-3.")
            H.diagonal().add_(1e-3)
            L     = torch.linalg.cholesky(H)
            H_inv = torch.cholesky_inverse(L)

        # ---- SIC quantisation loop ----
        # Process columns in blocks for cache efficiency (block_size = 128).
        # Inside each block, columns are processed one-by-one;
        # quantisation error is propagated to remaining columns via H^{-1}.
        W_q = torch.zeros_like(W)                  # output: quantised weights

        for blk_start in range(0, self.cols, self.block_size):
            blk_end  = min(blk_start + self.block_size, self.cols)
            blk_len  = blk_end - blk_start

            # Slice out block sub-matrices
            W_blk     = W[:, blk_start:blk_end].clone()    # (rows, blk)
            Hinv_blk  = H_inv[blk_start:blk_end,           # (blk, blk)
                               blk_start:blk_end]

            E_blk = torch.zeros_like(W_blk)                # error matrix for this block

            for j_local in range(blk_len):
                j_global = blk_start + j_local

                # Current weight column (all output neurons, one input feature)
                w_col = W_blk[:, j_local]                  # (rows,)

                # --- WaterSIC key step: use per-column eps or bits ---
                # Paper Section 3 / Alg. 1:
                #   w_hat_j = Q_{eps_j}(w_j)  with eps_j from water-filling
                if self.use_eps_grid:
                    eps_j   = col_eps[j_global].item()
                    # Rescale eps to weight magnitude for practical use
                    w_range = (w_col.max() - w_col.min()).item()
                    eps_abs = max(eps_j * w_range, 1e-8)
                    w_col_q = quantize_eps(w_col, eps_abs)
                else:
                    bits_j  = int(col_bits[j_global].item())
                    w_col_q = quantize_int(w_col, bits_j)

                W_q[:, j_global] = w_col_q

                # Quantisation error for this column
                delta = w_col - w_col_q                    # (rows,)

                # Scale error by 1/H_inv[j,j] for SIC update
                h_inv_jj = Hinv_blk[j_local, j_local]      # scalar
                err_scaled = delta / (h_inv_jj + 1e-12)    # (rows,)
                E_blk[:, j_local] = err_scaled

                # SIC: propagate error to remaining columns within block
                # w_k <- w_k - err_scaled * H_inv[j, k]  for k > j
                # Paper Algorithm 1, step "Update remaining weights"
                if j_local + 1 < blk_len:
                    W_blk[:, j_local + 1:] -= (
                        err_scaled.unsqueeze(1)                       # (rows, 1)
                        * Hinv_blk[j_local, j_local + 1:].unsqueeze(0)  # (1, rest)
                    )

            # Propagate block error to all subsequent columns (cross-block SIC)
            if blk_end < self.cols:
                W[:, blk_end:] -= E_blk @ H_inv[blk_start:blk_end, blk_end:]

        # ---- Undo activation-order permutation ----
        if self.actorder:
            W_q        = W_q[:, inv_perm]
            col_bits   = col_bits[inv_perm]
            col_eps    = col_eps[inv_perm]

        # ---- Write back quantised weights in-place ----
        self.layer.weight.data = W_q.to(orig_dtype)

        info = {
            "col_bits":        col_bits.tolist(),
            "col_eps":         col_eps.tolist(),
            "actual_mean_bits": float(col_bits.float().mean()),
            "per_column":      per_column_alloc,
        }
        return W_q.to(orig_dtype), info

    def reset(self) -> None:
        """Clear accumulated Hessian statistics (for multi-layer pipelines)."""
        self.H.zero_()
        self.n_samp = 0


print("WaterSICQuantizer class defined successfully.")

---

## 5. Comparison Baseline — Round-to-Nearest (RTN)

The simplest PTQ baseline: uniformly round every weight to the nearest value on an integer grid, completely ignoring activation statistics. This is the *no-Hessian, no-SIC* baseline the paper compares against.

In [ ]:
def rtn_quantize(
    layer: nn.Linear,
    bits: int,
) -> torch.Tensor:
    """
    Round-to-Nearest (RTN) quantisation — uniform baseline.

    No Hessian, no SIC, no calibration data: just round every weight
    independently to the nearest integer grid point.

    Paper Section 2: this is the simplest PTQ strategy, against which
    GPTQ and WaterSIC both improve significantly.

    Args:
        layer : nn.Linear (modified in-place).
        bits  : integer bit-width (e.g. 4).

    Returns:
        W_q : quantised weight tensor.
    """
    W = layer.weight.data.clone().float()
    W_q = quantize_int(W, bits)
    layer.weight.data = W_q.to(layer.weight.dtype)
    return W_q


def gptq_quantize(
    layer: nn.Linear,
    X: torch.Tensor,
    bits: int,
    damp_percent: float = 0.01,
    block_size: int = 128,
) -> torch.Tensor:
    """
    GPTQ baseline: SIC loop with *uniform* grid spacing (no water-filling).

    This is implemented by calling WaterSICQuantizer with
    per_column_alloc=False, which forces a constant eps / constant bits
    for all columns — exactly the GPTQ behaviour.

    Paper Section 5 / Appendix: WaterSIC generalises GPTQ by replacing
    the fixed epsilon with a per-column epsilon from water-filling.

    Args:
        layer    : nn.Linear (modified in-place).
        X        : calibration activations (N, in_features).
        bits     : integer bit-width (uniform).
        ...

    Returns:
        W_q : quantised weight tensor.
    """
    q = WaterSICQuantizer(layer, damp_percent=damp_percent,
                          block_size=block_size, actorder=True)
    q.add_batch(X)
    W_q, _ = q.quantize(target_avg_bits=float(bits), per_column_alloc=False)
    return W_q


print("Baseline functions defined.")

---

## 6. Evaluation Helper

We measure **reconstruction MSE** = $\|Wx - \hat{W}x\|^2_F / \|Wx\|^2_F$ on a held-out evaluation set — the same metric optimised by all three algorithms.

In [ ]:
@torch.no_grad()
def reconstruction_mse(
    W_orig: torch.Tensor,
    W_q:    torch.Tensor,
    X_eval: torch.Tensor,
) -> float:
    """
    Compute normalised reconstruction MSE on evaluation activations:
        rel_err = || (W - W_q) @ X_eval.T ||^2_F  /  || W @ X_eval.T ||^2_F

    Paper objective (Section 2): min_{W_hat} E[ || W x - W_hat x ||^2 ]

    Args:
        W_orig, W_q : (out, in) weight matrices.
        X_eval      : (N, in) evaluation activations.

    Returns:
        Relative MSE (float, lower is better).
    """
    X   = X_eval.float()
    out_orig = X @ W_orig.float().t()    # (N, out)
    out_q    = X @ W_q.float().t()       # (N, out)
    num = (out_orig - out_q).pow(2).sum().item()
    den = out_orig.pow(2).sum().item() + 1e-12
    return num / den


print("Evaluation helper defined.")

---

## 7. Synthetic Test — Bit-Width Sweep

We construct a synthetic `nn.Linear` layer with deliberately **non-uniform** activation variance across input channels (using `torch.linspace`). This makes the water-filling gain visible: if all channels had equal variance, water-filling degenerates to uniform allocation.

**Expected outcome (paper Section 3 / Fig. 1):**
- WaterSIC ≤ GPTQ ≤ RTN (in reconstruction error) at every bit-width
- The gap WaterSIC vs GPTQ widens as channel variances become more skewed
- All methods improve monotonically with more bits

In [ ]:
# ----------------------------------------------------------------
# Experiment configuration
# ----------------------------------------------------------------
IN_FEATURES  = 256
OUT_FEATURES = 128
N_CALIB      = 512     # calibration samples
N_EVAL       = 1024    # evaluation samples
VARIANCE_RATIO = 100.0 # ratio between largest and smallest channel variance
                       # larger ratio => bigger water-filling advantage

BIT_LEVELS = [1, 2, 3, 4, 5, 6, 8]

torch.manual_seed(SEED)

# Non-uniform activation variance: sigma_i in [sigma_min, sigma_max]
# This simulates real LLM activations where some input features are
# much more active than others (paper motivation, Section 1).
sigma = torch.linspace(0.1, 0.1 * VARIANCE_RATIO, IN_FEATURES).to(DEVICE)

X_calib = torch.randn(N_CALIB, IN_FEATURES, device=DEVICE) * sigma
X_eval  = torch.randn(N_EVAL,  IN_FEATURES, device=DEVICE) * sigma

print(f"Calibration set: {X_calib.shape}")
print(f"Evaluation set:  {X_eval.shape}")
print(f"Channel variance range: [{sigma.min():.2f}², {sigma.max():.2f}²]")
print(f"AM/GM variance ratio:   {(sigma**2).mean() / (sigma**2).prod()**(1/IN_FEATURES):.1f}x")

In [ ]:
# ----------------------------------------------------------------
# Run all three methods across all bit-widths
# ----------------------------------------------------------------
results = {"RTN": [], "GPTQ": [], "WaterSIC": []}

def fresh_layer():
    """Create a fresh nn.Linear for each trial (same random seed)."""
    torch.manual_seed(SEED)
    l = nn.Linear(IN_FEATURES, OUT_FEATURES, bias=False).to(DEVICE)
    nn.init.normal_(l.weight, std=0.02)
    return l


print(f"{'Bits':>5} | {'RTN rel-err':>14} | {'GPTQ rel-err':>14} | {'WaterSIC rel-err':>17} | {'GPTQ/WaterSIC':>14}")
print("-" * 75)

for bits in BIT_LEVELS:
    # ---- RTN baseline ----
    layer_rtn = fresh_layer()
    W_orig    = layer_rtn.weight.data.clone()
    rtn_quantize(layer_rtn, bits=bits)
    err_rtn   = reconstruction_mse(W_orig, layer_rtn.weight.data, X_eval)

    # ---- GPTQ baseline (uniform SIC) ----
    layer_gptq = fresh_layer()
    q_gptq     = WaterSICQuantizer(layer_gptq, actorder=True)
    q_gptq.add_batch(X_calib)
    _, info_gptq = q_gptq.quantize(
        target_avg_bits=float(bits), per_column_alloc=False
    )
    err_gptq = reconstruction_mse(W_orig, layer_gptq.weight.data, X_eval)

    # ---- WaterSIC ----
    layer_wsic = fresh_layer()
    q_wsic     = WaterSICQuantizer(layer_wsic, actorder=True)
    q_wsic.add_batch(X_calib)
    _, info_wsic = q_wsic.quantize(
        target_avg_bits=float(bits), per_column_alloc=True
    )
    err_wsic = reconstruction_mse(W_orig, layer_wsic.weight.data, X_eval)

    results["RTN"].append(err_rtn)
    results["GPTQ"].append(err_gptq)
    results["WaterSIC"].append(err_wsic)

    ratio = err_gptq / (err_wsic + 1e-12)
    print(f"{bits:>5}b | {err_rtn:>14.6f} | {err_gptq:>14.6f} | "
          f"{err_wsic:>17.6f} | {ratio:>12.2f}x")

print("-" * 75)
print("\nWaterSIC ≤ GPTQ: ",
      all(w <= g + 1e-9 for w, g in zip(results["WaterSIC"], results["GPTQ"])))
print("GPTQ ≤ RTN:      ",
      all(g <= r + 1e-9 for g, r in zip(results["GPTQ"], results["RTN"])))

## 8. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {"RTN": "#E74C3C", "GPTQ": "#3498DB", "WaterSIC": "#27AE60"}
markers = {"RTN": "s", "GPTQ": "^", "WaterSIC": "o"}

# ---- Plot 1: Relative MSE vs Bit-width ----
ax = axes[0]
for method in ["RTN", "GPTQ", "WaterSIC"]:
    ax.semilogy(
        BIT_LEVELS, results[method],
        label=method, color=colors[method],
        marker=markers[method], linewidth=2, markersize=7,
    )
ax.set_xlabel("Target bit-width", fontsize=12)
ax.set_ylabel("Relative reconstruction MSE", fontsize=12)
ax.set_title("Reconstruction Error vs Bit-Width", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, which="both", alpha=0.3)
ax.set_xticks(BIT_LEVELS)
ax.yaxis.set_major_formatter(mticker.LogFormatterSciNotation())

# Annotate the gap at 4 bits
idx4 = BIT_LEVELS.index(4)
ax.annotate(
    f"WaterSIC gap\n at 4 bits",
    xy=(4, results["WaterSIC"][idx4]),
    xytext=(5, results["WaterSIC"][idx4] * 3),
    arrowprops=dict(arrowstyle="->", color="#27AE60"),
    color="#27AE60", fontsize=9,
)

# ---- Plot 2: GPTQ/WaterSIC error ratio vs bit-width ----
ax2 = axes[1]
ratios = [
    g / (w + 1e-12)
    for g, w in zip(results["GPTQ"], results["WaterSIC"])
]
bars = ax2.bar(
    BIT_LEVELS, ratios,
    color=["#3498DB" if r >= 1 else "#E74C3C" for r in ratios],
    edgecolor="white", linewidth=0.8,
)
ax2.axhline(1.0, color="black", linestyle="--", linewidth=1.5, label="No gain")
for bar, ratio in zip(bars, ratios):
    ax2.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
        f"{ratio:.2f}×", ha="center", va="bottom", fontsize=9, fontweight="bold"
    )
ax2.set_xlabel("Bit-width", fontsize=12)
ax2.set_ylabel("GPTQ MSE / WaterSIC MSE  (>1 is WaterSIC win)", fontsize=11)
ax2.set_title("WaterSIC Gain Over GPTQ", fontsize=13, fontweight="bold")
ax2.legend(fontsize=11)
ax2.set_xticks(BIT_LEVELS)
ax2.grid(axis="y", alpha=0.3)

plt.suptitle(
    f"WaterSIC vs Baselines — IN={IN_FEATURES}, OUT={OUT_FEATURES}, "
    f"variance ratio={VARIANCE_RATIO:.0f}×",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.savefig("watersic_reconstruction_error.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as watersic_reconstruction_error.png")

## 9. Water-filling Visualisation — Bit Allocation Per Channel

In [ ]:
# Show how water-filling allocates bits as a function of channel importance
torch.manual_seed(SEED)

n_vis = 64                                # show first 64 channels
H_vis = compute_hessian(X_calib[:, :n_vis], damp_percent=0.01)
h_diag_vis = H_vis.diagonal()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ---- Left: Hessian diagonal (channel importance) ----
axes[0].bar(range(n_vis), h_diag_vis.cpu().numpy(), color="#3498DB", alpha=0.8)
axes[0].set_xlabel("Channel index", fontsize=11)
axes[0].set_ylabel("H diagonal value (importance λᵢ)", fontsize=11)
axes[0].set_title("Channel Importance from Hessian", fontsize=12, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)

# ---- Right: Water-filling bit allocation ----
target_bits_vis = 4.0
bits_vis, eps_vis = waterfilling_allocate(h_diag_vis, target_bits_vis, b_min=1, b_max=8)

# Also show uniform allocation
uniform_bits_vis = int(round(target_bits_vis))
cmap_vals = ["#E74C3C" if b < uniform_bits_vis else
             "#27AE60" if b > uniform_bits_vis else
             "#3498DB"
             for b in bits_vis.tolist()]

axes[1].bar(range(n_vis), bits_vis.cpu().numpy(), color=cmap_vals, alpha=0.85)
axes[1].axhline(uniform_bits_vis, color="black", linestyle="--",
                linewidth=2, label=f"Uniform ({uniform_bits_vis}b)")
axes[1].set_xlabel("Channel index", fontsize=11)
axes[1].set_ylabel("Allocated bits", fontsize=11)
axes[1].set_title(
    f"Water-filling Bit Allocation (target avg = {target_bits_vis}b)",
    fontsize=12, fontweight="bold"
)
axes[1].legend(fontsize=10)
axes[1].set_yticks(range(1, 9))
axes[1].grid(axis="y", alpha=0.3)

# Add legend patches
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#27AE60", label="> uniform (more bits)"),
    Patch(facecolor="#3498DB", label="= uniform"),
    Patch(facecolor="#E74C3C", label="< uniform (fewer bits)"),
]
axes[1].legend(handles=legend_elements, fontsize=9, loc="upper left")

plt.suptitle(
    "Water-filling allocates more bits to high-importance channels",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("watersic_bit_allocation.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Actual mean bits: {bits_vis.float().mean():.2f}  (target: {target_bits_vis})")

## 10. AM–GM Theoretical Gap Analysis

This cell reproduces the theoretical distortion gap from the paper (Section 3). The ratio $\bar{\lambda}_A / \bar{\lambda}_G$ predicts the maximum gain WaterSIC can achieve over GPTQ.

In [ ]:
# Compute AM-GM gap from calibration Hessian
H_full   = compute_hessian(X_calib, damp_percent=0.01)
lam_full = torch.linalg.eigvalsh(H_full).clamp(min=1e-9)  # eigenvalues of H

lam_A = lam_full.mean()                              # arithmetic mean
lam_G = lam_full.log().mean().exp()                  # geometric mean
am_gm_ratio = (lam_A / lam_G).item()
bit_gap = 0.5 * math.log2(am_gm_ratio)              # GPTQ excess bits over IT limit

print("Theoretical Analysis (Paper Section 3)")
print("=" * 50)
print(f"  Arithmetic mean λ_A = {lam_A.item():.4f}")
print(f"  Geometric  mean λ_G = {lam_G.item():.6f}")
print(f"  AM/GM ratio         = {am_gm_ratio:.2f}x")
print(f"  Predicted bit gap   = {bit_gap:.3f} bits")
print()
print("  D_GPTQ / D_WaterSIC ≈ λ_A / λ_G")
print(f"  => WaterSIC reduces distortion by ~{am_gm_ratio:.1f}x over GPTQ")
print(f"  => Equivalent to ~{bit_gap:.2f} extra bits in GPTQ to match WaterSIC")
print()

# WaterSIC's guaranteed gap to IT limit (paper Theorem)
it_gap_watersic = 0.5 * math.log2(2 * math.pi * math.e / 12)  # ≈ 0.255 bits
print(f"  WaterSIC gap to IT limit: ≤ {it_gap_watersic:.3f} bits (paper Theorem 1)")

## 11. End-to-End Sensitivity Analysis — Variance Ratio Effect

The AM–GM gap (and thus WaterSIC's advantage) grows with the **skewness** of the eigenvalue distribution. Here we sweep the variance ratio and plot how the WaterSIC gain scales.

In [ ]:
variance_ratios = [1, 2, 5, 10, 20, 50, 100, 500]
gains_4bit = []
am_gm_gaps = []

for vr in variance_ratios:
    torch.manual_seed(SEED)
    sig = torch.linspace(0.1, 0.1 * vr, IN_FEATURES).to(DEVICE)
    Xc  = torch.randn(N_CALIB, IN_FEATURES, device=DEVICE) * sig
    Xe  = torch.randn(N_EVAL,  IN_FEATURES, device=DEVICE) * sig

    # Theoretical AM-GM gap
    lam = (sig ** 2)                           # channel variances as proxy
    ratio = (lam.mean() / lam.log().mean().exp()).item()
    am_gm_gaps.append(0.5 * math.log2(max(ratio, 1.0)))

    # Empirical gain at 4 bits
    l_gptq = nn.Linear(IN_FEATURES, OUT_FEATURES, bias=False).to(DEVICE)
    nn.init.normal_(l_gptq.weight, std=0.02)
    W0 = l_gptq.weight.data.clone()
    l_wsic = nn.Linear(IN_FEATURES, OUT_FEATURES, bias=False).to(DEVICE)
    l_wsic.weight.data = W0.clone()

    q1 = WaterSICQuantizer(l_gptq, actorder=True)
    q1.add_batch(Xc)
    q1.quantize(target_avg_bits=4.0, per_column_alloc=False)

    q2 = WaterSICQuantizer(l_wsic, actorder=True)
    q2.add_batch(Xc)
    q2.quantize(target_avg_bits=4.0, per_column_alloc=True)

    eg = reconstruction_mse(W0, l_gptq.weight.data, Xe)
    ew = reconstruction_mse(W0, l_wsic.weight.data, Xe)
    gains_4bit.append(eg / (ew + 1e-12))


fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(variance_ratios, gains_4bit,
            "o-", color="#27AE60", linewidth=2, markersize=8, label="Empirical gain (4b)")
ax.semilogx(variance_ratios,
            [2 ** (2 * g) for g in am_gm_gaps],
            "--", color="#3498DB", linewidth=2, label="Theoretical AM/GM ratio")
ax.axhline(1.0, color="grey", linestyle=":")
ax.set_xlabel("Channel variance ratio (max σ² / min σ²)", fontsize=12)
ax.set_ylabel("Error ratio GPTQ / WaterSIC  (>1 = WaterSIC wins)", fontsize=11)
ax.set_title(
    "WaterSIC Advantage Grows with Channel Variance Skewness\n"
    "(matches paper Section 3: gap given by AM–GM inequality)",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=11)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("watersic_variance_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 12. Summary Table

In [ ]:
print("╔══════════════════╦══════════════════════╦═══════════════════════╦═══════════════╗")
print("║ Property         ║ RTN                  ║ GPTQ                  ║ WaterSIC      ║")
print("╠══════════════════╬══════════════════════╬═══════════════════════╬═══════════════╣")
print("║ Hessian          ║ None                 ║ H = X^T X             ║ H = X^T X     ║")
print("║ SIC update       ║ No                   ║ Yes                   ║ Yes           ║")
print("║ Bit allocation   ║ Uniform              ║ Uniform               ║ Water-filling ║")
print("║ Grid spacing ε   ║ Global               ║ Global                ║ Per-column    ║")
print("║ Distortion       ║ ∝ λ_A (worst)        ║ ∝ λ_A (GPTQ limit)   ║ ∝ λ_G (near  ║")
print("║                  ║                      ║                       ║  IT optimal)  ║")
print("║ IT gap           ║ Unbounded            ║ ½ log₂(λ_A/λ_G) bits ║ ≤ 0.255 bits  ║")
print("║ Calibration      ║ Not needed           ║ Required              ║ Required      ║")
print("║ Overhead         ║ O(1)                 ║ O(n²) Cholesky        ║ O(n²) + WF    ║")
print("╚══════════════════╩══════════════════════╩═══════════════════════╩═══════════════╝")

print()
print("Key equation (paper Section 3):")
print("  ε_i = sqrt(τ / λ_i)")
print("  where τ is the water level satisfying mean(R_i) = R_avg")
print("  and   R_i = ½ log₂(λ_i / τ)  [clipped to [b_min, b_max]]")
print()
print("Distortion comparison (paper Theorem, high-rate regime):")
print("  D_RTN  ≈ D_GPTQ ≈ λ̄_A · 2^{-2R}    (arithmetic mean)")
print("  D_WaterSIC ≈ λ̄_G · 2^{-2R}          (geometric mean)")
print("  D_IT*      = λ̄_G · 2^{-2R}          (IT lower bound)")
print("  => WaterSIC gap to IT: ≤ 0.255 bits  (paper Theorem 1)")

---

## References

1. **Lifar, Savkin, Ordentlich, Polyanskiy** — *WaterSIC: information-theoretically (near) optimal linear layer quantization*, arXiv:2603.04956v1, March 2026.

2. **Frantar et al.** — *GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers*, arXiv:2210.17323, 2022.

3. **Cover & Thomas** — *Elements of Information Theory*, Chapter 10: Water-filling.

4. **Ordentlich, Polyanskiy** — *High-Rate Quantized Matrix Multiplication: Theory and Practice*, arXiv:2601.17187, 2026.